# 02b — Sri Lanka holiday calendar

Build a small, auditable calendar of statutory public holidays for 2023–2024. The annual schedules are transcribed from the Government Gazettes; the one-off public holiday on 2024-09-23 is recorded separately from its later Gazette order. Bank-only holidays are excluded. Poya is a separate flag; the day following Vesak is not itself flagged as Poya.

Sources: [Gazette Extraordinary No. 2287/04 (2023 schedule)](https://documents.gov.lk/view/extra-gazettes/2022/7/2287-04_E.pdf); [Gazette Extraordinary No. 2341/46 (2024 schedule)](https://documents.gov.lk/view/extra-gazettes/2023/7/2341-46_E.pdf); [Gazette Extraordinary No. 2402/25 (2024-09-23 special public holiday)](https://documents.gov.lk/view/extra-gazettes/2024/9/2402-25_E.pdf). A Central Bank calendar is available as an official cross-check for annual dates.

The output is a date-level calendar, not a complete date spine. Join it to the household-day data on the local Sri Lankan calendar date, not a UTC date. The final section rereads the saved CSV and displays counts by year, Poya dates, duplicate dates, and missing values.

In [ ]:
from pathlib import Path
import csv
from datetime import date
import pandas as pd

# Supports running from either the project root or the notebooks/ directory.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
OUTPUT_PATH = PROJECT_ROOT / "data" / "interim" / "sri_lanka_holidays_2023_2024.csv"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Calendar output: {OUTPUT_PATH}")

In [ ]:
# Official annual schedules; one row per holiday date.
HOLIDAY_ROWS = [
    {"date": "2023-01-06", "holiday_name": "Duruthu Full Moon Poya Day", "is_public_holiday": True, "is_poya_day": True, "source_reference": "Gazette Extraordinary No. 2287/04 (2022-07-04)"},
    {"date": "2023-01-15", "holiday_name": "Tamil Thai Pongal Day", "is_public_holiday": True, "is_poya_day": False, "source_reference": "Gazette Extraordinary No. 2287/04 (2022-07-04)"},
    {"date": "2023-02-04", "holiday_name": "Independence Day", "is_public_holiday": True, "is_poya_day": False, "source_reference": "Gazette Extraordinary No. 2287/04 (2022-07-04)"},
    {"date": "2023-02-05", "holiday_name": "Nawam Full Moon Poya Day", "is_public_holiday": True, "is_poya_day": True, "source_reference": "Gazette Extraordinary No. 2287/04 (2022-07-04)"},
    {"date": "2023-02-18", "holiday_name": "Maha Sivarathri Day", "is_public_holiday": True, "is_poya_day": False, "source_reference": "Gazette Extraordinary No. 2287/04 (2022-07-04)"},
    {"date": "2023-03-06", "holiday_name": "Madin Full Moon Poya Day", "is_public_holiday": True, "is_poya_day": True, "source_reference": "Gazette Extraordinary No. 2287/04 (2022-07-04)"},
    {"date": "2023-04-05", "holiday_name": "Bak Full Moon Poya Day", "is_public_holiday": True, "is_poya_day": True, "source_reference": "Gazette Extraordinary No. 2287/04 (2022-07-04)"},
    {"date": "2023-04-07", "holiday_name": "Good Friday", "is_public_holiday": True, "is_poya_day": False, "source_reference": "Gazette Extraordinary No. 2287/04 (2022-07-04)"},
    {"date": "2023-04-13", "holiday_name": "Day Prior to Sinhala & Tamil New Year Day", "is_public_holiday": True, "is_poya_day": False, "source_reference": "Gazette Extraordinary No. 2287/04 (2022-07-04)"},
    {"date": "2023-04-14", "holiday_name": "Sinhala & Tamil New Year Day", "is_public_holiday": True, "is_poya_day": False, "source_reference": "Gazette Extraordinary No. 2287/04 (2022-07-04)"},
    {"date": "2023-04-22", "holiday_name": "Id-Ul-Fitr (Ramazan Festival Day)", "is_public_holiday": True, "is_poya_day": False, "source_reference": "Gazette Extraordinary No. 2287/04 (2022-07-04)"},
    {"date": "2023-05-01", "holiday_name": "May Day", "is_public_holiday": True, "is_poya_day": False, "source_reference": "Gazette Extraordinary No. 2287/04 (2022-07-04)"},
    {"date": "2023-05-05", "holiday_name": "Vesak Full Moon Poya Day", "is_public_holiday": True, "is_poya_day": True, "source_reference": "Gazette Extraordinary No. 2287/04 (2022-07-04)"},
    {"date": "2023-05-06", "holiday_name": "Day Following Vesak Full Moon Poya Day", "is_public_holiday": True, "is_poya_day": False, "source_reference": "Gazette Extraordinary No. 2287/04 (2022-07-04)"},
    {"date": "2023-06-03", "holiday_name": "Poson Full Moon Poya Day", "is_public_holiday": True, "is_poya_day": True, "source_reference": "Gazette Extraordinary No. 2287/04 (2022-07-04)"},
    {"date": "2023-06-29", "holiday_name": "Id-Ul-Alha (Hadji Festival Day)", "is_public_holiday": True, "is_poya_day": False, "source_reference": "Gazette Extraordinary No. 2287/04 (2022-07-04)"},
    {"date": "2023-07-03", "holiday_name": "Adhi Esala Full Moon Poya Day", "is_public_holiday": True, "is_poya_day": True, "source_reference": "Gazette Extraordinary No. 2287/04 (2022-07-04)"},
    {"date": "2023-08-01", "holiday_name": "Esala Full Moon Poya Day", "is_public_holiday": True, "is_poya_day": True, "source_reference": "Gazette Extraordinary No. 2287/04 (2022-07-04)"},
    {"date": "2023-08-30", "holiday_name": "Nikini Full Moon Poya Day", "is_public_holiday": True, "is_poya_day": True, "source_reference": "Gazette Extraordinary No. 2287/04 (2022-07-04)"},
    {"date": "2023-09-28", "holiday_name": "Milad-Un-Nabi (Holy Prophet's Birthday)", "is_public_holiday": True, "is_poya_day": False, "source_reference": "Gazette Extraordinary No. 2287/04 (2022-07-04)"},
    {"date": "2023-09-29", "holiday_name": "Binara Full Moon Poya Day", "is_public_holiday": True, "is_poya_day": True, "source_reference": "Gazette Extraordinary No. 2287/04 (2022-07-04)"},
    {"date": "2023-10-28", "holiday_name": "Vap Full Moon Poya Day", "is_public_holiday": True, "is_poya_day": True, "source_reference": "Gazette Extraordinary No. 2287/04 (2022-07-04)"},
    {"date": "2023-11-12", "holiday_name": "Deepavali Festival Day", "is_public_holiday": True, "is_poya_day": False, "source_reference": "Gazette Extraordinary No. 2287/04 (2022-07-04)"},
    {"date": "2023-11-26", "holiday_name": "Ill Full Moon Poya Day", "is_public_holiday": True, "is_poya_day": True, "source_reference": "Gazette Extraordinary No. 2287/04 (2022-07-04)"},
    {"date": "2023-12-25", "holiday_name": "Christmas Day", "is_public_holiday": True, "is_poya_day": False, "source_reference": "Gazette Extraordinary No. 2287/04 (2022-07-04)"},
    {"date": "2023-12-26", "holiday_name": "Unduvap Full Moon Poya Day", "is_public_holiday": True, "is_poya_day": True, "source_reference": "Gazette Extraordinary No. 2287/04 (2022-07-04)"},
    {"date": "2024-01-15", "holiday_name": "Tamil Thai Pongal Day", "is_public_holiday": True, "is_poya_day": False, "source_reference": "Gazette Extraordinary No. 2341/46 (2023-07-20)"},
    {"date": "2024-01-25", "holiday_name": "Duruthu Full Moon Poya Day", "is_public_holiday": True, "is_poya_day": True, "source_reference": "Gazette Extraordinary No. 2341/46 (2023-07-20)"},
    {"date": "2024-02-04", "holiday_name": "Independence Day", "is_public_holiday": True, "is_poya_day": False, "source_reference": "Gazette Extraordinary No. 2341/46 (2023-07-20)"},
    {"date": "2024-02-23", "holiday_name": "Nawam Full Moon Poya Day", "is_public_holiday": True, "is_poya_day": True, "source_reference": "Gazette Extraordinary No. 2341/46 (2023-07-20)"},
    {"date": "2024-03-08", "holiday_name": "Maha Sivarathri Day", "is_public_holiday": True, "is_poya_day": False, "source_reference": "Gazette Extraordinary No. 2341/46 (2023-07-20)"},
    {"date": "2024-03-24", "holiday_name": "Medin Full Moon Poya Day", "is_public_holiday": True, "is_poya_day": True, "source_reference": "Gazette Extraordinary No. 2341/46 (2023-07-20)"},
    {"date": "2024-03-29", "holiday_name": "Good Friday", "is_public_holiday": True, "is_poya_day": False, "source_reference": "Gazette Extraordinary No. 2341/46 (2023-07-20)"},
    {"date": "2024-04-11", "holiday_name": "Id-Ul-Fitr (Ramazan Festival Day)", "is_public_holiday": True, "is_poya_day": False, "source_reference": "Gazette Extraordinary No. 2341/46 (2023-07-20)"},
    {"date": "2024-04-12", "holiday_name": "Day Prior to Sinhala & Tamil New Year Day", "is_public_holiday": True, "is_poya_day": False, "source_reference": "Gazette Extraordinary No. 2341/46 (2023-07-20)"},
    {"date": "2024-04-13", "holiday_name": "Sinhala & Tamil New Year Day", "is_public_holiday": True, "is_poya_day": False, "source_reference": "Gazette Extraordinary No. 2341/46 (2023-07-20)"},
    {"date": "2024-04-23", "holiday_name": "Bak Full Moon Poya Day", "is_public_holiday": True, "is_poya_day": True, "source_reference": "Gazette Extraordinary No. 2341/46 (2023-07-20)"},
    {"date": "2024-05-01", "holiday_name": "May Day (International Workers' Day)", "is_public_holiday": True, "is_poya_day": False, "source_reference": "Gazette Extraordinary No. 2341/46 (2023-07-20)"},
    {"date": "2024-05-23", "holiday_name": "Vesak Full Moon Poya Day", "is_public_holiday": True, "is_poya_day": True, "source_reference": "Gazette Extraordinary No. 2341/46 (2023-07-20)"},
    {"date": "2024-05-24", "holiday_name": "Day Following Vesak Full Moon Poya Day", "is_public_holiday": True, "is_poya_day": False, "source_reference": "Gazette Extraordinary No. 2341/46 (2023-07-20)"},
    {"date": "2024-06-17", "holiday_name": "Id-Ul-Alha (Hadji Festival Day)", "is_public_holiday": True, "is_poya_day": False, "source_reference": "Gazette Extraordinary No. 2341/46 (2023-07-20)"},
    {"date": "2024-06-21", "holiday_name": "Poson Full Moon Poya Day", "is_public_holiday": True, "is_poya_day": True, "source_reference": "Gazette Extraordinary No. 2341/46 (2023-07-20)"},
    {"date": "2024-07-20", "holiday_name": "Esala Full Moon Poya Day", "is_public_holiday": True, "is_poya_day": True, "source_reference": "Gazette Extraordinary No. 2341/46 (2023-07-20)"},
    {"date": "2024-08-19", "holiday_name": "Nikini Full Moon Poya Day", "is_public_holiday": True, "is_poya_day": True, "source_reference": "Gazette Extraordinary No. 2341/46 (2023-07-20)"},
    {"date": "2024-09-16", "holiday_name": "Milad-Un-Nabi (Holy Prophet's Birthday)", "is_public_holiday": True, "is_poya_day": False, "source_reference": "Gazette Extraordinary No. 2341/46 (2023-07-20)"},
    {"date": "2024-09-17", "holiday_name": "Binara Full Moon Poya Day", "is_public_holiday": True, "is_poya_day": True, "source_reference": "Gazette Extraordinary No. 2341/46 (2023-07-20)"},
    {"date": "2024-09-23", "holiday_name": "Special Public Holiday", "is_public_holiday": True, "is_poya_day": False, "source_reference": "Gazette Extraordinary No. 2402/25 (2024-09-22)"},
    {"date": "2024-10-17", "holiday_name": "Vap Full Moon Poya Day", "is_public_holiday": True, "is_poya_day": True, "source_reference": "Gazette Extraordinary No. 2341/46 (2023-07-20)"},
    {"date": "2024-10-31", "holiday_name": "Deepawali Festival Day", "is_public_holiday": True, "is_poya_day": False, "source_reference": "Gazette Extraordinary No. 2341/46 (2023-07-20)"},
    {"date": "2024-11-15", "holiday_name": "Ill Full Moon Poya Day", "is_public_holiday": True, "is_poya_day": True, "source_reference": "Gazette Extraordinary No. 2341/46 (2023-07-20)"},
    {"date": "2024-12-14", "holiday_name": "Unduvap Full Moon Poya Day", "is_public_holiday": True, "is_poya_day": True, "source_reference": "Gazette Extraordinary No. 2341/46 (2023-07-20)"},
    {"date": "2024-12-25", "holiday_name": "Christmas Day", "is_public_holiday": True, "is_poya_day": False, "source_reference": "Gazette Extraordinary No. 2341/46 (2023-07-20)"},
]

EXPECTED_COLUMNS = ["date", "holiday_name", "is_public_holiday", "is_poya_day", "source_reference"]
holidays = pd.DataFrame(HOLIDAY_ROWS, columns=EXPECTED_COLUMNS)
holidays["date"] = pd.to_datetime(holidays["date"], format="%Y-%m-%d", errors="raise")
holidays = holidays.sort_values("date").reset_index(drop=True)

# QA: schema, unique dates, non-empty names/references, years, and logical flags.
assert holidays.columns.tolist() == EXPECTED_COLUMNS
assert holidays["date"].is_unique, "More than one row exists for a holiday date."
assert holidays["holiday_name"].notna().all() and holidays["holiday_name"].str.strip().ne("").all()
assert holidays["source_reference"].notna().all() and holidays["source_reference"].str.strip().ne("").all()
assert holidays["date"].dt.year.isin([2023, 2024]).all()
assert holidays["is_public_holiday"].all(), "Calendar should contain public holidays only."
assert not ((holidays["is_poya_day"]) & (~holidays["is_public_holiday"])).any()
assert len(holidays) == 52, f"Expected 52 public-holiday dates; found {len(holidays)}."
assert holidays["is_poya_day"].sum() == 25, "Unexpected Poya-day count; re-check Gazette transcription."

holidays["date"] = holidays["date"].dt.strftime("%Y-%m-%d")
holidays.to_csv(OUTPUT_PATH, index=False, encoding="utf-8", quoting=csv.QUOTE_MINIMAL)
print(f"Wrote {len(holidays)} rows to {OUTPUT_PATH}")

In [ ]:
# Read the saved CSV back and independently check the persisted artifact.
saved = pd.read_csv(OUTPUT_PATH, dtype={"date": "string"})
assert saved.columns.tolist() == EXPECTED_COLUMNS
assert len(saved) == 52 and saved["date"].is_unique
assert saved.notna().all().all()
assert saved["date"].str.match(r"^202[34]-\d{2}-\d{2}$").all()
print("Rows by year:")
display(saved.assign(year=saved["date"].str[:4]).groupby("year").agg(
    holidays=("date", "size"), poya_days=("is_poya_day", lambda s: (s.astype(str).str.lower() == "true").sum())
))
print("Special one-off holiday included:", saved.loc[saved["date"] == "2024-09-23", ["date", "holiday_name", "source_reference"]].to_dict("records"))
display(saved.head(10))

## Persisted calendar validation summary

The summary below is computed from the CSV after it has been written and read back. It reports annual holiday counts, Poya dates, duplicate dates, and missing values, and checks those values against the expected 2023–2024 calendar contract.


In [ ]:
# Summarize the persisted artifact, including all counts requested for the calendar audit.
saved_year = saved["date"].str.slice(0, 4)
year_counts = saved.assign(year=saved_year).groupby("year").size().to_dict()
poya_mask = saved["is_poya_day"].astype("string").str.lower().eq("true")
text_columns = ["date", "holiday_name", "source_reference"]
blank_text_values = sum(
    int(saved[column].astype("string").str.strip().eq("").fillna(False).sum())
    for column in text_columns
)
missing_value_count = int(saved.isna().sum().sum()) + blank_text_values
duplicate_date_count = int(saved["date"].duplicated().sum())

calendar_validation_summary = pd.DataFrame([
    ("Total holiday dates", len(saved), 52),
    ("2023 holiday dates", int(year_counts.get("2023", 0)), 26),
    ("2024 holiday dates", int(year_counts.get("2024", 0)), 26),
    ("Poya dates", int(poya_mask.sum()), 25),
    ("Duplicate dates", duplicate_date_count, 0),
    ("Missing values", missing_value_count, 0),
], columns=["measure", "observed", "expected"])
calendar_validation_summary["passed"] = calendar_validation_summary["observed"].eq(
    calendar_validation_summary["expected"]
)
display(calendar_validation_summary)
failed_calendar_checks = calendar_validation_summary.loc[
    ~calendar_validation_summary["passed"], "measure"
].tolist()
if failed_calendar_checks:
    raise RuntimeError(f"Holiday calendar validation failed: {failed_calendar_checks}")
print("HOLIDAY CALENDAR VALIDATION COMPLETE")
